In [1]:
# Instalación silenciosa de dependencias necesarias
!pip install -qU langchain-chroma langchain-huggingface sentence-transformers pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/6

In [2]:
from google.colab import drive
import os

# Montar Drive
drive.mount('/content/drive')

# Definir rutas exactas del proyecto
WORK_DIR = '/content/drive/MyDrive/Proyecto_RAG_Coyuntura'
DB_DIR = os.path.join(WORK_DIR, 'chroma_db')

print(f"Directorio de base de datos configurado en: {DB_DIR}")

Mounted at /content/drive
Directorio de base de datos configurado en: /content/drive/MyDrive/Proyecto_RAG_Coyuntura/chroma_db


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Inicializar exactamente el mismo modelo de embeddings de tu motor
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Conectar a la base de datos persistida
vectorstore = Chroma(
    persist_directory=DB_DIR,
    embedding_function=embeddings
)

K = 5  # Recuperar los top 5 documentos (igual que en motor_rag.py)
retriever = vectorstore.as_retriever(search_kwargs={"k": K})

print(f"Vectorstore cargado exitosamente. Documentos totales indexados: {vectorstore._collection.count()}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore cargado exitosamente. Documentos totales indexados: 9422


In [39]:
TEST_SET = [
    # --- NIVEL 1: BÚSQUEDAS FACTUALES Y ENTIDADES ---
    {
        "pregunta": "¿Qué medidas de compensación económica y operativa anunció YPFB por los daños causados por la 'gasolina basura'?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/esta-es-la-linea-de-whatsapp-de-ypfb-para-reportar-danos-por-gasolina-desestabilizada"]
    },
    {
        "pregunta": "¿Qué refugio de animales denunció que la vida de sus especies rescatadas estaba en riesgo debido al bloqueo en La Cumbre?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/senda-verde-alerta-que-no-tienen-comida-para-alimentar-casi-1-200-animales-por-los-bloqueos"]
    },
    {
        "pregunta": "¿Qué dijo el ministro de Economía, José Gabriel Espinoza, sobre el rol de la inversión privada en la movilidad social?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/la-inversion-privada-debe-generar-movilidad-social-y-resultados-concretos-dice-espinoza-en"]
    },
    {
        "pregunta": "¿Cuáles fueron los motivos iniciales por los que la Central Obrera Boliviana (COB) inició sus movilizaciones antes de pedir la renuncia presidencial?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/la-cob-se-desvincula-de-evo-dice-que-ya-tuvo-oportunidad-y-niega-financiamiento"]
    },
    {
        "pregunta": "¿A qué gran acuerdo convocó Revilla durante la Sesión de Honor de la Asamblea?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/la-camara-de-diputados-aprueba-proyecto-que-abroga-la-ley-1720"]
    },

    # --- NIVEL 2: BÚSQUEDAS CON FILTRO TEMPORAL (MESES) ---
    {
        "pregunta": "¿Qué sectores exigieron la renuncia del presidente Rodrigo Paz mediante bloqueos durante el mes de mayo?",
        "mes_filtro": "mayo",
        "urls_relevantes": ["https://www.urgente.bo/noticia/salazar-advierte-que-paz-tiene-dos-caminos-renunciar-o-enfrentar-una-convulsion-social"]
    },
    {
        "pregunta": "¿A quiénes acusó directamente el presidente Rodrigo Paz de promover los bloqueos carreteros en sus declaraciones de julio?",
        "mes_filtro": "julio",
        "urls_relevantes": ["https://www.urgente.bo/noticia/en-medio-de-los-conflictos-tuto-y-paz-enfrentan-una-ruptura-segun-analistas"]
    },
    {
        "pregunta": "¿Qué propuestas presentó el Gobierno a los choferes durante las negociaciones de marzo para evaluar la calidad del combustible?",
        "mes_filtro": "marzo",
        "urls_relevantes": ["https://www.urgente.bo/noticia/ya-se-distribuye-gasolina-de-buena-calidad-los-choferes-preparan-comision-fiscalizadora"]
    },
    {
        "pregunta": "¿Qué postura expresaron los legisladores en enero respecto al trabajo y retraso de elecciones según las críticas a Rodrigo Paz?",
        "mes_filtro": "enero",
        "urls_relevantes": ["https://www.urgente.bo/noticia/me-pone-enojado-samuel-cuestiona-paz-por-demora-en-normas-y-pide-que-se-debatan-publicamente"]
    },
    {
        "pregunta": "¿Qué reportó la Administradora Boliviana de Carreteras (ABC) sobre el estado de las vías tras 53 días de conflictos?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/paz-el-bloqueo-ha-sido-derrotado-no-puede-retornar-al-pais"]
    },

    # --- NIVEL 3: ANÁLISIS DE COYUNTURA Y SÍNTESIS TÉCNICA ---
    {
        "pregunta": "¿Cuáles son las consecuencias principales del fenómeno del 'Súper Niño' y el cambio climático en el altiplano boliviano según el investigador Roger Carvajal?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/el-super-nino-y-el-cambio-climatico-adelantan-el-crudo-invierno-en-bolivia-e-impulsan-la"]
    },
    {
        "pregunta": "¿Por qué los expertos en clima e hidrología advierten que Bolivia enfrentará 'sequías extremas' a partir del año 2027?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/el-super-nino-y-el-cambio-climatico-adelantan-el-crudo-invierno-en-bolivia-e-impulsan-la"]
    },
    {
        "pregunta": "¿Cuál es el escenario que enfrenta Bolivia respecto a la importación de gas natural y qué advierten los analistas energéticos?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/zaratti-bolivia-tendra-que-importar-gas-dentro-de-tres-anos-si-no-se-reduce-el-consumo"]
    },
    {
        "pregunta": "¿Qué impacto económico diario proyectaron los expertos financieros a raíz del anuncio de cierre de rutas internacionales por parte del transporte pesado?",
        "mes_filtro": None,
        "urls_relevantes": ["https://www.urgente.bo/noticia/gobierno-y-empresarios-privados-acuerdan-agenda-para-reactivar-el-comercio-exterior-tras"]
    },
    {
        "pregunta": "¿Qué prácticas y medidas de mitigación están implementando los productores agropecuarios para ser más resilientes ante la crisis hídrica y las heladas fuera de temporada?",
        "mes_filtro": None,
        "urls_relevantes": ["urgente.bo/noticia/justiniano-presenta-presenta-el-plan-estratégico-2026–2030-la-cooperación-internacional"]
    }
]

print(f"Preguntas de prueba cargadas: {len(TEST_SET)}")

Preguntas de prueba cargadas: 15


In [40]:
import pandas as pd

def precision_recall_at_k(recuperados_urls, relevantes_urls):
    recuperados_set = set(recuperados_urls)
    relevantes_set = set(relevantes_urls)
    aciertos = recuperados_set & relevantes_set

    precision = len(aciertos) / len(recuperados_set) if recuperados_set else 0.0
    recall = len(aciertos) / len(relevantes_set) if relevantes_set else 0.0
    return precision, recall

def reciprocal_rank(recuperados_urls_ordenados, relevantes_urls):
    relevantes_set = set(relevantes_urls)
    for rank, url in enumerate(recuperados_urls_ordenados, start=1):
        if url in relevantes_set:
            return 1.0 / rank
    return 0.0

resultados = []
for caso in TEST_SET:
    search_kwargs = {"k": K}
    if caso["mes_filtro"]:
        search_kwargs["filter"] = {"mes": caso["mes_filtro"].lower()}

    retriever.search_kwargs = search_kwargs
    docs = retriever.invoke(caso["pregunta"])

    # Extraer URLs recuperadas (evitando duplicados si se traen varios chunks de la misma noticia)
    urls_ordenadas = list(dict.fromkeys(d.metadata["url"] for d in docs))

    precision, recall = precision_recall_at_k(urls_ordenadas, caso["urls_relevantes"])
    rr = reciprocal_rank(urls_ordenadas, caso["urls_relevantes"])

    resultados.append({
        "pregunta": caso["pregunta"],
        "mes_filtro": caso["mes_filtro"],
        f"precision@{K}": round(precision, 3),
        f"recall@{K}": round(recall, 3),
        "reciprocal_rank": round(rr, 3),
        "urls_recuperadas": urls_ordenadas,
    })

df_resultados = pd.DataFrame(resultados)
df_resultados

,pregunta,mes_filtro,precision@5,recall@5,reciprocal_rank,urls_recuperadas
0,¿Qué medidas de compensación económica y opera...,NaN,0.000,0.0,0.0,[https://www.urgente.bo/noticia/esta-es-la-l%C...
1,¿Qué refugio de animales denunció que la vida ...,NaN,0.000,0.0,0.0,[https://www.urgente.bo/noticia/senda-verde-bl...
2,"¿Qué dijo el ministro de Economía, José Gabrie...",NaN,0.000,0.0,0.0,[https://www.urgente.bo/noticia/%E2%80%9Cla-in...
3,¿Cuáles fueron los motivos iniciales por los q...,NaN,0.200,1.0,1.0,[https://www.urgente.bo/noticia/la-cob-se-desv...
4,¿A qué gran acuerdo convocó Revilla durante la...,NaN,0.200,1.0,1.0,[https://www.urgente.bo/noticia/la-camara-de-d...
5,¿Qué sectores exigieron la renuncia del presid...,mayo,0.000,0.0,0.0,[https://www.urgente.bo/noticia/la-cob-dialoga...
6,¿A quiénes acusó directamente el presidente Ro...,julio,0.000,0.0,0.0,[https://www.urgente.bo/noticia/bloqueos-civic...
7,¿Qué propuestas presentó el Gobierno a los cho...,marzo,0.200,1.0,1.0,[https://www.urgente.bo/noticia/ya-se-distribu...
8,¿Qué postura expresaron los legisladores en en...,enero,0.000,0.0,0.0,[https://www.urgente.bo/noticia/pdc-y-libre-ad...
9,¿Qué reportó la Administradora Boliviana de Ca...,NaN,0.200,1.0,1.0,[https://www.urgente.bo/noticia/paz-el-bloqueo...


In [41]:
resumen = {
    f"Precision@{K} promedio": round(df_resultados[f"precision@{K}"].mean(), 3),
    f"Recall@{K} promedio": round(df_resultados[f"recall@{K}"].mean(), 3),
    "MRR (Mean Reciprocal Rank) promedio": round(df_resultados["reciprocal_rank"].mean(), 3)
}

df_resumen = pd.DataFrame([resumen])
df_resumen

,Precision@5 promedio,Recall@5 promedio,MRR (Mean Reciprocal Rank) promedio
0,0.116,0.533,0.533


In [42]:
# Exportacion de resultados
df_resultados.to_excel(os.path.join(WORK_DIR, "evaluacion/resultados.xlsx"), index=False)
df_resumen.to_excel(os.path.join(WORK_DIR, "evaluacion/resumen.xlsx"), index=False)